# Insurance Customer Risk Profiling — revised methodology

**Academic group project (2025) | AXA insurance underwriting context**

This is a **new reproducible methodological revision**, not the same experiment that generated the historical performance table. It has **not been run on the original authorized dataset**, and makes **no new performance claim**.

**Data access:** An authorized dataset is required at `data/Insurance Data.xlsx` (not included). Features are processed after splitting labeled observations into training and validation sets. Numeric/categorical preprocessing and SMOTE operate only within the training fold using an `imblearn` Pipeline. `OneHotEncoder` replaces arbitrary integer codes for nominal categories. The unlabeled rows are never used to fit preprocessing or evaluate metrics.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

DATA_PATH = Path('data/Insurance Data.xlsx')
TARGET = 'Risk Profile'
RANDOM_STATE = 42
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Authorized dataset not found: {DATA_PATH}')

In [ ]:
df = pd.read_excel(DATA_PATH)
if TARGET not in df:
    raise ValueError(f'Missing column: {TARGET}')
known_labels = ['Low', 'Medium', 'High']
labeled = df[df[TARGET].isin(known_labels)].copy()
unlabeled = df[df[TARGET].isna()].copy()
print('Labeled:',len(labeled),'Unlabeled:',len(unlabeled))
print(labeled[TARGET].value_counts())
# Never use customer identifiers as predictors.
ID_COLS = [c for c in labeled.columns if c.strip().lower() in {'customer id','customer_id','id'}]
X = labeled.drop(columns=[TARGET]+ID_COLS)
y = labeled[TARGET]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print('Train:', X_train.shape, 'Validation:', X_val.shape)

## Time-derived and ratio features

Use a deterministic feature transformer without fitting to validation statistics. Invalid divisions (zero denominator) are set to missing and then handled by training-fitted imputers. Policy dates are dropped after deriving duration/time components; raw dates are not used as categorical strings.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class RiskFeatureBuilder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        d = X.copy()
        for col in ['Policy Start Date', 'Policy Renewal Date']:
            if col in d.columns:
                d[col] = pd.to_datetime(d[col], errors='coerce')
        if {'Policy Start Date','Policy Renewal Date'}.issubset(d.columns):
            d['Policy_Duration'] = (d['Policy Renewal Date']-d['Policy Start Date']).dt.days
            d['Start_Year'] = d['Policy Start Date'].dt.year
            d['Start_Month'] = d['Policy Start Date'].dt.month
        for dest,a,b in [
            ('Premium_Coverage_Ratio','Premium Amount','Coverage Amount'),
            ('Deductible_Coverage_Ratio','Deductible','Coverage Amount'),
            ('Premium_Age_Ratio','Premium Amount','Age'),
        ]:
            if a in d and b in d:
                den = pd.to_numeric(d[b],errors='coerce').replace(0,np.nan)
                d[dest] = pd.to_numeric(d[a],errors='coerce')/den
        if 'Age' in d and 'Credit Score' in d:
            d['Age_Credit_Interaction'] = pd.to_numeric(d['Age'],errors='coerce') * pd.to_numeric(d['Credit Score'],errors='coerce')
        d = d.drop(columns=['Policy Start Date','Policy Renewal Date'], errors='ignore')
        # Guard against future dates or other datetimes remaining as raw predictors.
        d = d.drop(columns=d.select_dtypes(include=['datetime','datetimetz']).columns)
        return d.replace([np.inf,-np.inf],np.nan)

builder = RiskFeatureBuilder()
X_train_fe = builder.transform(X_train)
num_cols = X_train_fe.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train_fe.columns if c not in num_cols]
print('Numeric features:',len(num_cols),'Categorical:',len(cat_cols))

## Leakage-safe pipeline

The imputer and encoder are fit on training data only. For a mixed feature matrix after one-hot encoding, `RandomOverSampler` is used for the **revised experiment** rather than SMOTE on arbitrary ordinal-encoded nominal categories. This is a deliberate methodological change, not reproduction of historical SMOTE results. The pipeline is refit independently inside each CV fold.

In [ ]:
num_pipe = SkPipeline([('impute', SimpleImputer(strategy='median'))])
cat_pipe = SkPipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
prep = ColumnTransformer([
    ('numeric',num_pipe,num_cols),('categorical',cat_pipe,cat_cols)
], remainder='drop')

model = Pipeline([
    ('feature_engineering', RiskFeatureBuilder()),
    ('preprocessing', prep),
    ('resampling', RandomOverSampler(random_state=RANDOM_STATE)),
    ('classifier', RandomForestClassifier(n_estimators=200,random_state=RANDOM_STATE,n_jobs=-1))
])
# This example keeps the model fixed; tune only within train-side CV if needed.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
metrics = cross_validate(model, X_train, y_train, cv=cv, scoring={'accuracy':'accuracy','f1_macro':'f1_macro'}, n_jobs=1)
print('CV accuracy:',metrics['test_accuracy'].mean())
print('CV macro-F1:',metrics['test_f1_macro'].mean())

In [ ]:
model.fit(X_train, y_train)
pred = model.predict(X_val)
print(classification_report(y_val, pred,labels=known_labels,zero_division=0))
print('Confusion matrix (Low, Medium, High):')
print(confusion_matrix(y_val,pred,labels=known_labels))
print('Validation accuracy:',accuracy_score(y_val,pred))
print('Validation macro-F1:',f1_score(y_val,pred,average='macro'))
print('Validation F1 (High):',f1_score(y_val,pred,labels=['High'],average='macro'))

## Interpretation and next checks

- Compare against a majority-class and simple classifier baseline; verify the **Medium** class recall, not just overall accuracy.
- Run with the authorized dataset to obtain *new*, distinct performance figures; do not substitute historical numbers.
- Verify temporal availability of every feature at underwriting decision time (potential target leakage), label validity and representativeness, fairness across population groups, and business review before deployment.
- If the data represent fictional/synthetic individuals, describe performance as a synthetic exercise only. Do not claim real-world underwriting impact.